<a href="https://colab.research.google.com/github/yuri-bizzoni/yuribizzoni/blob/ml-humanities-notebooks/Felicita_in_tre_domande.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tre domande sulla felicità

Un ricordo felice può durare una riga. Chi dice «noi»? Di chi è il successo? Se si ricorda una cena, che cosa è successo davvero a tavola?

Nel 2017 i partecipanti a HappyDB hanno scritto in inglese brevi risposte a due consegne. Una chiedeva che cosa li avesse resi felici nelle ultime 24 ore, l'altra negli ultimi tre mesi. Leggeremo questi testi nella lingua originale.

Esegui le due celle iniziali. Poi puoi scegliere uno qualsiasi dei tre esercizi. Servono Python, pandas e scikit-learn. Se necessario, installali in una cella con `%pip install pandas scikit-learn`.

[Presentazione del corpus e dei campi](https://github.com/megagonlabs/HappyDB) · [Asai et al., LREC 2018](https://aclanthology.org/L18-1103/)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

url = "https://raw.githubusercontent.com/megagonlabs/HappyDB/master/happydb/data/cleaned_hm.csv"
campi = ["hmid", "wid", "reflection_period", "cleaned_hm", "ground_truth_category"]
dati = pd.read_csv(url, usecols=campi).dropna(subset=["wid", "cleaned_hm"])
print(f"{len(dati):,} racconti disponibili")

100,535 racconti disponibili


In [ ]:
# Le prime tre righe permettono di vedere che cosa ha scritto ogni persona.
for _, riga in dati.head(3).iterrows():
    print(f"[{riga['reflection_period']}] {riga['cleaned_hm']}")

[24h] I went on a successful date with someone I felt sympathy and connection with.
[24h] I was happy when my son got 90% marks in his examination 
[24h] I went to the gym this morning and did yoga.


## 1. Regressione lineare · Quando diciamo «noi»?

**Chi ripensa a tre mesi racconta la felicità al plurale più spesso di quando ripensa a ieri?** Contiamo le forme inglesi della prima persona plurale ogni cento parole. Per confrontare le stesse voci, teniamo chi ha risposto a entrambe le consegne. Ogni persona pesa uguale, anche se ha scritto molti racconti.

La retta ha solo due valori sull'asse orizzontale. Zero indica le 24 ore e uno i tre mesi. La sua pendenza è la differenza media fra le due consegne, espressa in forme plurali ogni cento parole.

In [ ]:
from sklearn.linear_model import LinearRegression

testi = dati["cleaned_hm"].str.lower()
conteggi = dati[["wid", "reflection_period"]].assign(
    plurali=testi.str.count(r"\b(?:we|us|our|ours|ourselves)\b"),
    parole=testi.str.count(r"\b[a-z]+(?:['’][a-z]+)*\b")
)
totali = conteggi.groupby(["wid", "reflection_period"])[["plurali", "parole"]].sum()
frequenze = 100 * totali["plurali"] / totali["parole"]
coppie = frequenze.unstack()[["24h", "3m"]].dropna()

osservazioni = coppie.melt(var_name="periodo", value_name="plurali_per_100")
X = osservazioni[["periodo"]].eq("3m").astype(int)
retta = LinearRegression().fit(X, osservazioni["plurali_per_100"])
print(f"{len(coppie)} persone hanno risposto a entrambe le consegne.")
print(f"24 ore: {retta.intercept_:.2f}    3 mesi: {retta.intercept_ + retta.coef_[0]:.2f}")
print(f"Differenza per 100 parole: {retta.coef_[0]:+.2f}")

3008 persone hanno risposto a entrambe le consegne.
24 ore: 0.70    3 mesi: 0.88
Differenza per 100 parole: +0.18


**Da discutere.** Immagina due persone che raccontano la stessa passeggiata. Una scrive *I walked home with my sister*, l’altra *We walked home together*. Solo la seconda fa salire il conteggio. Che cosa ci dice la differenza sulla voce narrante? Che cosa possiamo sapere della compagnia in cui si trovava?

## 2. SVM lineare · Un successo può essere anche una forma d’affetto?

**Quando un genitore racconta un traguardo del figlio, quali parole fanno leggere l'episodio come affetto, e quali come riuscita?** Impariamo la distinzione sull'insieme dei racconti etichettati dal corpus come *affection* o *achievement*. Poi cerchiamo i casi familiari che mettono in difficoltà questa distinzione.

TF–IDF dà peso alle parole che caratterizzano un racconto. La SVM impara quali favoriscono una delle due letture. La prova finale usa persone diverse da quelle incontrate durante l'addestramento.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

categoria = "ground_truth_category"
annotati = dati[dati[categoria].isin(["affection", "achievement"])].copy()
annotati = annotati.drop_duplicates("cleaned_hm")
divisione = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
indici_train, indici_test = next(divisione.split(annotati, groups=annotati["wid"]))
train, test = annotati.iloc[indici_train], annotati.iloc[indici_test].copy()

svm = make_pipeline(TfidfVectorizer(min_df=2), LinearSVC(random_state=42))
svm.fit(train["cleaned_hm"], train[categoria])
test["prevista"] = svm.predict(test["cleaned_hm"])
esatte = test["prevista"].eq(test[categoria]).mean()
prevalente = train[categoria].mode().iat[0]
base = test[categoria].eq(prevalente).mean()
print(f"Su {len(test)} racconti di prova: SVM {esatte:.1%}, categoria più frequente {base:.1%}.")

Su 2127 racconti di prova: SVM 95.3%, categoria più frequente 51.3%.


In [ ]:
parole = svm[0].get_feature_names_out()
pesi = svm[1].coef_[0].argsort()
for classe, indici in zip(svm.classes_, [pesi[:7], pesi[-7:][::-1]]):
    print(f"{classe}: {', '.join(parole[indici])}")

figli = test["cleaned_hm"].str.contains(r"\b(?:son|daughter)\b", case=False)
discordanti = test[figli & test["prevista"].ne(test[categoria])]
for _, caso in discordanti.head(3).iterrows():
    print(f"\n{caso['cleaned_hm']}")
    print(f"Corpus: {caso[categoria]} | SVM: {caso['prevista']}")

achievement: job, boss, exam, exams, raise, appreciated, car
affection: husband, son, family, wife, daughter, my, mom

I went to my son's hockey game and they won 4-3.  It was a great comeback being down 3-0 after one period.
Corpus: affection | SVM: achievement

My son got a job offer when attending the job fair at school, many of the employers were impressed with his JROTC uniform. 
Corpus: affection | SVM: achievement

What made me really happy is that my daughter got a job offer for a great position she really wanted.
Corpus: affection | SVM: achievement


**Da discutere.** Scegli uno dei disaccordi. Sottolinea il gesto o il risultato e cerchia la parola che nomina un legame. Quale delle due etichette daresti al racconto se potessi sceglierne solo una?

## 3. K-means · Che cosa ricordiamo di una cena felice?

**Due cene si somigliano perché si mangia bene, perché si cucina per qualcuno o perché ci si ritrova?** Selezioniamo i racconti che contengono *dinner* e togliamo quella parola dal vocabolario. K-means raggruppa i testi in base alle parole rimaste. I gruppi partono senza nome.

Per leggere il risultato, guardiamo prima le parole con il peso più alto nel centro di ciascun gruppo. Poi confrontiamo due racconti vicini al centro con uno scelto a caso nello stesso gruppo.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

cene = dati.loc[dati["cleaned_hm"].str.contains(r"\bdinner\b", case=False), "cleaned_hm"]
cene = cene.drop_duplicates().reset_index(drop=True)
escluse = sorted(ENGLISH_STOP_WORDS | {"dinner"})
lessico = TfidfVectorizer(stop_words=escluse, min_df=4)
X_cene = lessico.fit_transform(cene)

# Alcune risposte, come "Dinner!", perdono tutte le parole utilizzabili.
utili = X_cene.getnnz(axis=1) > 0
cene = cene[utili].reset_index(drop=True)
X_cene = X_cene[utili]
k = 4
gruppi = KMeans(n_clusters=k, n_init=10, random_state=42)
etichette = gruppi.fit_predict(X_cene)
distanze = gruppi.transform(X_cene)
print(f"{len(cene)} racconti di cene, {k} gruppi")

3251 racconti di cene, 4 gruppi


In [ ]:
parole_cene = lessico.get_feature_names_out()
for gruppo, centro in enumerate(gruppi.cluster_centers_):
    termini = parole_cene[centro.argsort()[-6:][::-1]]
    membri = cene[etichette == gruppo]
    vicini = distanze[membri.index, gruppo].argsort()[:2]
    print(f"\nGruppo {gruppo + 1} · {len(membri)} racconti · {', '.join(termini)}")
    for frase in membri.iloc[vicini]:
        print("  Vicino al centro:", frase)
    altro = membri.drop(membri.index[vicini]).sample(1, random_state=42).iat[0]
    print("  A caso:", altro)


Gruppo 1 · 2083 racconti · husband, delicious, ate, happy, night, good
  Vicino al centro: I ate a delicious dinner with my husband.
  Vicino al centro: Ate a delicious dinner last night.
  A caso: I made a good dinner last night, and I have left overs in the fridge to eat later.

Gruppo 2 · 559 racconti · went, friends, friend, nice, restaurant, time
  Vicino al centro: I went out to dinner with my friends. 
  Vicino al centro: I went out with my friends to dinner. 
  A caso: My girlfriend and I went to dinner. 

Gruppo 3 · 246 racconti · wife, cooked, went, got, nice, home
  Vicino al centro: I had dinner out with my wife.
  Vicino al centro: my wife made dinner.
  A caso: Last night my wife and I went out to dinner with some of our friends, and we told them we were pregnant!

Gruppo 4 · 363 racconti · family, went, nice, happy, enjoyed, time
  Vicino al centro: my hole family out to have dinner
  Vicino al centro: Had dinner with family. 
  A caso: I celebrated my birthday with fam

**Da discutere.** Dai a ogni gruppo un titolo usando un verbo, per esempio *ritrovarsi*. Il titolo funziona anche per il racconto scelto a caso? Prova a cambiare `k` da 4 a 3, riesegui le due celle e osserva quali distinzioni scompaiono.